# Cell 1 — Imports & config


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, to_date, regexp_extract, explode_outer, trim
)
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()
CATALOG = "company_risk_intelligence_platform"

# Cell 2 — Inspect bronze (understand the shape)


In [0]:
df_peek = spark.read.table(f"{CATALOG}.bronze.ch_filing_history")
df_peek.printSchema()
display(df_peek.limit(5))

# Cell 3 — SCD1 merge helper
# 

In [0]:
def scd_merge(source_df, target_table, business_key):
    if not spark.catalog.tableExists(target_table):
        print(f"First load -> {target_table}")
        (source_df.write.format("delta")
            .mode("overwrite")
            .saveAsTable(target_table))
        print("Table created")
    else:
        print(f"Incremental SCD1 merge -> {target_table}")
        tgt = DeltaTable.forName(spark, target_table)
        cond = " AND ".join([f"t.{k} = s.{k}" for k in business_key])
        (tgt.alias("t")
            .merge(source_df.alias("s"), cond)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
        print("Merge completed")

# Cell 4 — Transformation function


In [0]:
def transform_ch_people(df):

    # 1. Extract company_number from the file path
    df = df.withColumn(
        "company_number",
        regexp_extract(col("file_path"), r"/([A-Za-z0-9]+)_\d{8}_\d{6}\.json$", 1)
    )

    # 2. Explode the officers array -> one row per officer
    df = df.withColumn("officer", explode_outer(col("items")))

    # 3. Select & flatten (only fields that exist in your schema)
    df = df.select(
        col("company_number"),
        trim(col("officer.name")).alias("officer_name"),
        col("officer.officer_role").alias("officer_role"),
        to_date(col("officer.appointed_on")).alias("appointed_on"),
        to_date(col("officer.resigned_on")).alias("resigned_on"),
        col("officer.nationality").alias("nationality"),
        col("officer.country_of_residence").alias("country_of_residence"),
        col("last_update_ts").alias("ingestion_ts"),
    )

    # 4. Derive active flag (no resignation date = currently active)
    df = df.withColumn("is_active", col("resigned_on").isNull())

    # 5. Drop rows with no officer name (defensive)
    df = df.filter(col("officer_name").isNotNull() & (col("officer_name") != ""))

    # 6. Deduplicate on the business key
    df = df.dropDuplicates(["company_number", "officer_name", "officer_role", "appointed_on"])

    return df

# Cell 5 — Run transform & preview


In [0]:
src = spark.read.table(f"{CATALOG}.bronze.ch_people")
silver_people = transform_ch_people(src)

print("Row count:", silver_people.count())
display(silver_people.limit(20))

#Cell 6 — Load into Silver

In [0]:
target_table = f"{CATALOG}.silver.ch_people"
business_key = ["company_number", "officer_name", "officer_role", "appointed_on"]
scd_merge(silver_people, target_table, business_key)

# Cell 6 — Validation


In [0]:
silver = spark.read.table(f"{CATALOG}.silver.ch_people")

print("Companies covered:")
display(silver.select("company_number").distinct().orderBy("company_number"))

print("Active vs resigned officers per company:")
display(
    silver.groupBy("company_number", "is_active").count()
          .orderBy("company_number", "is_active")
)

print("Officer role breakdown:")
display(silver.groupBy("officer_role").count().orderBy(col("count").desc()))